In [424]:
# in second video : we took 32K names dataset and used combination of 2 characters as to train and predict the names 
# 26 alphabets and additional '.' 
# we count each and every unique combinations starting from '..' to 'z.' : means 27 * 27 possibilities and the number of counts for each in the entire dataset
# as it is clearly visible that it is growing exponentially as we move from two char combos to 3 , the possible outcomes '...' to 'zz.' are 27 * 27 * 27

# https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf

# our approach was basically 2D : row * columns and one hot encoding for each character to uniquely represent
# the above paper uses the 30 dimension to represent each word and allows to group the words with same meaning to be close to each other : cat and dog are close to each other than cat and car : N-dimensional representation , this allows transfer learning 
# for an example if in case the data never contains "the dog is walking in ______ " , but it has seen "a cat is running in room." then here ['a' ,'the'] articles might be close , similarly ['cat' ,'dog'] might be close , ['walking','running'] might be close to each other and allows to transfer learning

In [425]:
import torch 
import torch.nn.functional as F 
import matplotlib.pyplot as plt

In [426]:
#reading the data
words = open('names.txt','r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [427]:
# total 32k+ names
len(words)

32033

In [428]:
# characters
chars = sorted(list(set(''.join(words))))
# embedding character to number
stoi = {s : i+1 for i , s in enumerate(chars)}
# special character for start/end
stoi['.'] = 0
# reverse embedding for each character , number to character
itos = {i : s for s, i in stoi.items()}



In [429]:
# the block size is the context length needed to predict the next word : here we want 3 characters to predict the 4th one
block_size = 3

# x are the inputs to the NN and y are the labels for each input
X, Y = [] , []

# working with first 5 words 
for w in words:
  
  # print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    # print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)


In [430]:
# the paper has 16K+ dataset and each is represented in 30 dimensions , here we have 32K+ names which we have mapped to 27 characters each is represented in slightly lower number of dimensions : 2 for now
# So : imagine first table that has all 27 characters and are index from 0 to 26 : from . to z 
#  then the C is (27,2) table where both column for each 27 characters has random values to start with
# now using one_hot() function in torch we connected the character table to C table : HOW ? 
# as we know for any given tensor value N and the dimension d : the one_hot will create the d (1,d) dimensional vector of 0s and the value at Nth position will be 1 
# for an example : F.one_hot(torch.tensor(5) , num_classes = 27 ) : [0 0 0 0 1 0 0 0 0.....0]
# now we also have to be careful as the above one_hot() vector is int in data type whereas the C table is float so we will convert the one_hot to float() and then multiply it with C (vector multiplication)
# what that does is : simply gives us the Values of C : as multiplying anything with 0s is 0 and with 1 is the number it self 
# F.one_hot(torch.tensor(5) , num_classes = 27 ).float()
# F.one_hot(torch.tensor(5) , num_classes = 27 ).float() @ W : [0 0 0 0 1 0 0 ..... 0] @ [[random_float_value_1 , random_float_value_2]] : [[random_float_value_1 , random_float_value_2]]



# The second way : Just embedding the values to C directly : C[0] , C[1] ....
# the way torch.randn() and the way torch is designed : it allows us to index the C[X] and it does the same as we did using the one_hot()
# All thanks to pytorch indexing






In [431]:
# # The two dimensional look up table for all 27 characters
# C = torch.randn((27,2))
# # C.shape : (27,2)



# # The complete embedding for all 27  characters :  two dimensional random value from C table
# emb = C[X]
# emb.shape

# # as we know the shape of our embedding are (32,3,2) 



# # The weights for NN 

# W1 = torch.randn((6, 100))
# # print(W1.shape) : (6,100)




# # biases 
# b1 = torch.randn(100)

# # the goal here is to do the W @ emb + b but the dimension for the emb is (32,2,2) where as the dimension for W1 is (6,100)
# # so we can't multiply the (32,3,2) tensor with (6,100)
# # to make (32,3,2) tensor (32,6) we use view() method : it allows to change the dimensions of vector by stacking/concatenation of dimension : the first 32 remains the same 

# # also as we know , this is the implementation of MLP where we will have multiple layers of NN : starting with hidden layer h : that is the reason why we have w1 and b1 and w and b for weights and bias

# # simply w*x + b with activation on tanh
# h = torch.tanh(emb.view(-1,6) @ W1 + b1)
# # h.shape : (32,100)




# # for the last layer : to predict the next word ,we have 27 possible characters to calculate from 
# # so we will create (100,27) dimensional weights and (1,27) dimensional biases for the last hidden layer

# W2 = torch.rand((100,27))
# # print(W2.shape) : (100.27)


# h2 = torch.rand((27))
# #h2.shape : 27 or  1 row and 27 columns

# # the final logits 
# logits = h @ W2 + h2 
# # logits.shape : (32,27)


# # Now exp() to converge the range of values 
# counts =logits.exp()
# #print(counts.shape) : (32,27)



# # finding the probabilities 
# prob = counts / counts.sum(1 , keepdim=True)
# #prob.shape : (32,27)

# # as we know the sum of each row in prob is 1 : meaning , for the given context : the sum of probability of all possible character as a next character is 1

# loss = -prob[torch.arange(32) , Y].log().mean()
# loss

In [432]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g  , requires_grad=True)
W1 = torch.randn((6, 100), generator=g  , requires_grad=True)
b1 = torch.randn(100, generator=g  , requires_grad=True)
W2 = torch.randn((100, 27), generator=g  , requires_grad=True)
b2 = torch.randn(27, generator=g  , requires_grad=True)
parameters = [C, W1, b1, W2, b2]

In [433]:
sum(parameter.nelement() for parameter in parameters)

3481

In [434]:
## Training the NN : 
# learning_rate_exp = torch.linspace(-3,0,1000)
# learning_rate = 10**learning_rate_exp
# lri = []
# lossi = []
for i in range(1000):

## Forward pass
    # As we tried to use the entire dataset at once and did decent on minimizing the loss , from 19 to around 2.6 in 100 iterations : we noticed it was slow 
    # the solution is to use the batches of data instead of the entire dataset : size 32 
    ix = torch.randint(0,X.shape[0] , (32,))
     

    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2 

# manually calculating the loss : slow , and has memory problem for extreme values , and creates new tensors for each operation : it is advice to use the cross_entropy() instead , 
# why cross_entropy ? : it controls the extreme values and avoids the overflowing , makes forward and backward pass easier , does not creates new tensors 
#####

# counts = logits.exp()
# prob = counts / counts.sum(1, keepdim=True)
# loss = -prob[torch.arange(32) , Y].log().mean()
# loss
####

    loss = F.cross_entropy(logits, Y[ix])
    

    # print(loss.item())


## backward pass 

    for p in parameters : 
        # Set weight grad to None / zero
        p.grad = None
    
    loss.backward()

    # update the gradient values
    # lr = learning_rate[i]
    lr = 0.1
    for p in parameters:
        p.data += -lr* p.grad



    # lri.append(learning_rate_exp[i])

    # lossi.append(loss.item())


print(loss.item())


2.658998727798462


In [ ]:
# The lesson in this tutorial is : we started with roughly 3K parameters and the data size is small , but in real life where we have multi-millions of parameters and relatively small set of data , we can achieve 0 loss or near 0 loss , that doesn't mean model is train and has better performance , it rather indicates that model has over-fit and memorized data : Model won't produce any new data , it will create the same results as available in dataset

# Training split 80% - to train the actual parameters of the model 
# Dev/Validation Split  10% - to train the hyper parameters of the model
# Test split 10%